# Phase 6: Feature Engineering (Pollutants Only)

## 🎯 Objective
Construct rich, strictly backward-looking feature representations from the historical pollutant time series without data leakage.

### Feature Sets Created:
1. **Current Pollutant Signals**: $PM_{2.5}, PM_{10}, NO_2, SO_2, CO, O_3, NH_3$, and computed EPA AQI at time $t$.
2. **Historical Lags**: $1, 3, 6, 12, 24$ hours past observations for $PM_{2.5}, PM_{10}, NO_2, O_3$, and EPA AQI.
3. **Backward Rolling Aggregates**: Rolling mean, std, min, and max over $6, 12, 24$ hour historical windows.
4. **Chemical Interaction Ratios**: $PM_{2.5}/PM_{10}$ ratio, $NO_2/O_3$ oxidation ratio, $CO/NO_2$ combustion index (zero-division safe).
5. **Differential Velocity**: $1$-hour and $24$-hour rate-of-change indicators.
6. **Cyclical Temporal Encodings**: $\sin/\cos$ harmonics for hour-of-day, day-of-week, and month-of-year, plus weekend indicators.

**Total Features**: 64 deterministically ordered columns saved to `data/processed/feature_schema.json`.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.feature_pipeline.feature_engineering import FeatureEngineeringPipeline

clean_csv = PROJECT_ROOT / "data" / "processed" / "historical_aqi_clean.csv"
df_clean = pd.read_csv(clean_csv)
print(f"Loaded clean dataset with {len(df_clean):,} records.")

## 1. Execute Feature Engineering Pipeline

In [ ]:
pipeline = FeatureEngineeringPipeline()
df_features, feature_names = pipeline.build_features(df_clean, drop_na=True)

schema_file = PROJECT_ROOT / "data" / "processed" / "feature_schema.json"
pipeline.save_feature_schema(schema_file)

print(f"\nEngineered Features Shape: {df_features.shape}")
print(f"Number of Features: {len(feature_names)}")
print("\nFirst 15 Features:")
for f in feature_names[:15]:
    print(f"  - {f}")

## 2. Inspect Feature Correlation with Target AQI

In [ ]:
target_corrs = df_features[feature_names].corrwith(df_features['epa_aqi']).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
top_corrs = pd.concat([target_corrs.head(10), target_corrs.tail(5)])
top_corrs.plot(kind='barh', color='#4A90D9', edgecolor='black')
plt.title('Top Feature Correlations with EPA AQI')
plt.xlabel('Pearson Correlation Coefficient (r)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 10 Most Correlated Features with Current AQI:")
display(target_corrs.head(10))

## 3. Verify Anti-Leakage Properties

To prove absence of future data leakage into features:
- Every lag feature strictly corresponds to $t - k$ ($k \ge 1$).
- Rolling statistics use backward-looking windows $[t-w+1 ... t]$.
- Temporal sine/cosine encodings are computed exclusively from the current index timestamp $t$.

In [ ]:
# Inspect a sample 24-hour snapshot showing lag and rolling relationships
sample_cols = ["epa_aqi", "epa_aqi_lag_1h", "epa_aqi_lag_24h", "epa_aqi_rolling_mean_6h", "pm_ratio", "hour_sin"]
display(df_features[sample_cols].iloc[50:60])

## 4. Key Conclusions for Phase 7 (Training Dataset Preparation)
- **Feature Matrix Ready**: 48,808 samples $\times$ 64 engineered features.
- **Zero Data Leakage**: All features are strictly backward-looking.
- **Saved Schema**: `feature_schema.json` locks the feature order for Phase 7 dataset building, Phase 8-10 model training, and Phase 12 inference.